In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F


MSA Encoder Block (like Evoformer MSA Stack)

In [3]:
class MSAEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim, num_heads)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.ReLU(),
            nn.Linear(embed_dim * 4, embed_dim)
        )
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        # x: (seq_len, batch_size, embed_dim)
        attn_out, _ = self.attn(x, x, x)
        x = self.norm1(x + attn_out)
        x = self.norm2(x + self.ff(x))
        return x


Pairwise Representation Module

In [4]:
class PairwiseUpdate(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.triangle = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim)
        )

    def forward(self, pair_repr):
        # Simplified triangle update
        b, L, _, d = pair_repr.shape
        concat = torch.cat([pair_repr, pair_repr.transpose(1, 2)], dim=-1)
        update = self.triangle(concat)
        return pair_repr + update


Invariant Point Attention (simplified)

In [5]:
class InvariantPointAttention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.linear = nn.Linear(embed_dim, 3)  # Predicts XYZ coordinates

    def forward(self, pair_repr):
        # Mean pooling and projection to 3D
        b, L, _, d = pair_repr.shape
        avg_repr = pair_repr.mean(dim=2)
        coords = self.linear(avg_repr)
        return coords


In [6]:
class SimpleAlphaFoldModel(nn.Module):
    def __init__(self, msa_dim=64, pair_dim=64, num_heads=4):
        super().__init__()
        self.msa_encoder = MSAEncoderBlock(msa_dim, num_heads)
        self.pair_update = PairwiseUpdate(pair_dim)
        self.struct_module = InvariantPointAttention(pair_dim)

    def forward(self, msa, pair_repr):
        # msa: (seq_len, batch, msa_dim)
        msa_out = self.msa_encoder(msa)
        pair_updated = self.pair_update(pair_repr)
        coords = self.struct_module(pair_updated)
        return coords


In [11]:
seq_len = 128
batch_size = 2
msa_dim = pair_dim = 64

msa_input = torch.rand(seq_len, batch_size, msa_dim)
pair_repr = torch.rand(batch_size, seq_len, seq_len, pair_dim)

model = SimpleAlphaFoldModel(msa_dim, pair_dim)
predicted_coords = model(msa_input, pair_repr)

print(predicted_coords.shape)  # [batch_size, seq_len, 3]


torch.Size([2, 128, 3])


In [10]:
print(msa_input)

tensor([[[0.5224, 0.6427, 0.1696,  ..., 0.3434, 0.0351, 0.2831],
         [0.6868, 0.5440, 0.3707,  ..., 0.4206, 0.3704, 0.2629]],

        [[0.5867, 0.9910, 0.7623,  ..., 0.5494, 0.7339, 0.9186],
         [0.7697, 0.1380, 0.2851,  ..., 0.5495, 0.6653, 0.5619]],

        [[0.4462, 0.9713, 0.0240,  ..., 0.2061, 0.5514, 0.0178],
         [0.2279, 0.2265, 0.0032,  ..., 0.3621, 0.7754, 0.9902]],

        ...,

        [[0.8472, 0.2322, 0.3965,  ..., 0.0947, 0.1775, 0.5731],
         [0.5407, 0.6833, 0.8382,  ..., 0.8418, 0.3551, 0.9818]],

        [[0.3183, 0.7085, 0.6115,  ..., 0.9826, 0.3191, 0.3338],
         [0.6958, 0.0147, 0.5952,  ..., 0.0574, 0.3600, 0.7038]],

        [[0.4244, 0.1066, 0.7001,  ..., 0.7872, 0.1493, 0.4292],
         [0.5233, 0.0148, 0.7980,  ..., 0.2868, 0.2000, 0.4756]]])


Using Data set to train the model, Example Format: 
msa_input: (seq_len, batch_size, msa_dim)
pair_repr: (batch_size, seq_len, seq_len, pair_dim)
target_coords: (batch_size, seq_len, 3)

In [ ]:
from torch.utils.data import Dataset

class RNADataset(Dataset):
    def __init__(self, data_list):
        self.data = data_list

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        msa, pair_repr, coords = self.data[idx]
        return msa, pair_repr, coords

In [ ]:
import torch.nn as nn

loss_fn = nn.MSELoss()

In [ ]:
from torch.utils.data import DataLoader

model = SimpleAlphaFoldModel()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

dataloader = DataLoader(RNADataset(your_data), batch_size=4, shuffle=True)

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for msa, pair_repr, coords in dataloader:
        optimizer.zero_grad()
        pred_coords = model(msa, pair_repr)
        loss = loss_fn(pred_coords, coords)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch}, Loss: {total_loss}")
